# Fashion Retrieval — Colab: kısmi fine-tune (Adım 5)

**Ne yapar:** SigLIP görüntü kulesinin son 4 bloğu + projeksiyon başlığı, train_fit kırpıntılarından InfoNCE ile eğitilir.
Her tur sonunda val_internal Recall@1 ölçülür; en iyi model Drive'a yazılır (`outputs/checkpoints/ft/`). Bitince en iyi modelle
val_internal ve test öznitelikleri Drive'a (`features/siglip_vitb16+ft/`) çıkarılır ve **çalışma zamanı kendiliğinden kapanır**.

**Tahmini süre (L4):** zip açma 5 dk + 8 tur × ~4 dk + öznitelik çıkarma 3 dk ≈ **40–45 dk**.

**Nasıl kullanılır:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 GPU**. Sonra **Çalışma zamanı → Tümünü çalıştır**
(Ctrl+F9) deyip bırakın. Drive bağlama penceresi ilk hücrede bir kez izin ister; onu onayladıktan sonra başka müdahale gerekmez.
Oturum düşerse aynı şekilde yeniden "Tümünü çalıştır": eğitim kaldığı turdan sürer (last.pt Drive'da).

In [ ]:
GITHUB_USER = "ethembelka"
REPO_NAME   = "fashion-product-retrieval"
DRIVE_ROOT  = "/content/drive/MyDrive/CMPE9103_Retrieval"
CROPS_ZIP   = f"{DRIVE_ROOT}/data/crops.zip"
PROCESSED   = "/content/processed"
CK_DIR      = f"{DRIVE_ROOT}/outputs/checkpoints/ft"
FEATURES    = f"{DRIVE_ROOT}/features/siglip_vitb16+ft"

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(CK_DIR, exist_ok=True); os.makedirs(FEATURES, exist_ok=True)
assert os.path.exists(CROPS_ZIP), f"crops.zip bulunamadı: {CROPS_ZIP}"
!nvidia-smi -L

In [ ]:
url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if os.path.exists(f"/content/{REPO_NAME}"):
    !git -C /content/{REPO_NAME} pull -q
else:
    !git clone -q {url} /content/{REPO_NAME}
%cd /content/{REPO_NAME}
!git log --oneline -1
!pip install -q timm open_clip_torch faiss-cpu pyarrow

In [ ]:
import time
if not os.path.exists(f"{PROCESSED}/crops"):
    t = time.time()
    !cp "{CROPS_ZIP}" /content/crops.zip
    !mkdir -p {PROCESSED} && unzip -q /content/crops.zip -d {PROCESSED} && rm /content/crops.zip
    print(f"açıldı, {time.time()-t:.0f} s")
!ls {PROCESSED}/splits

## Eğitim (uzun hücre — ilerleme satır satır yazılır)

In [ ]:
!python scripts/finetune.py --processed {PROCESSED} --out {CK_DIR} --features {FEATURES} --splits val_internal test

## Kontrol ve otomatik kapanış

In [ ]:
import json, glob
h = json.load(open(f"{CK_DIR}/history.json"))
for r in h: print(r)
print("features:", sorted(glob.glob(f"{FEATURES}/*/*_emb.npy")))
from google.colab import runtime
runtime.unassign()   # GPU'yu bırakır; sekme açık kalsa da süre harcanmaz